In [1]:
from mofapy2.run.entry_point import entry_point
import pandas as pd
import numpy as np
import os

# initialise the entry point
ent = entry_point()


        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
       
 
        


In [2]:
INPUT_FOLDER = "C:/Users/49152/Downloads/Multi-omics/SCOT_plus/AGW_results/"
OUTPUT_FOLDER = "C:/Users/49152/Downloads/Multi-omics/SCOT_plus/MOFA_output/"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

PROT_FILE = os.path.join(INPUT_FOLDER, "proteomics_for_MOFA.csv")
RNA_FILE = os.path.join(INPUT_FOLDER, "transcriptomics_for_MOFA.csv")
META_FILE = os.path.join(INPUT_FOLDER, "aligned_metadata.csv")

In [5]:
proteomics = pd.read_csv(PROT_FILE)
transcriptomics = pd.read_csv(RNA_FILE)

# Set first column name to Gene
proteomics.rename(columns={proteomics.columns[0]: "Gene"}, inplace=True)
transcriptomics.rename(columns={transcriptomics.columns[0]: "Gene"}, inplace=True)

# Set Gene as index
proteomics_mat = proteomics.set_index("Gene")
transcriptomics_mat = transcriptomics.set_index("Gene")

print("Proteomics matrix shape (features x samples):", proteomics_mat.shape)
print("Transcriptomics matrix shape (features x samples):", transcriptomics_mat.shape)

Proteomics matrix shape (features x samples): (1500, 68)
Transcriptomics matrix shape (features x samples): (1500, 68)


In [9]:
# Check pseudo-paired sample axix
prot_samples = pd.Index(proteomics_mat.columns)
rna_samples = pd.Index(transcriptomics_mat.columns)

print(f"Proteomics samples: {len(prot_samples)}")
print(f"Transcriptomics samples: {len(rna_samples)}")

if not prot_samples.equals(rna_samples):
    common_samples = prot_samples.intersection(rna_samples).sort_values()
    print("Warning: sample axes are not identical. Subsetting to common samples.")
    print(f"Common paired samples after SCOT+: {len(common_samples)}")

    if len(common_samples) == 0:
        raise ValueError("No shared SampleID values found between SCOT-aligned transcriptomics and proteomics matrices.")

    proteomics_mat = proteomics_mat.loc[:, common_samples]
    transcriptomics_mat = transcriptomics_mat.loc[:, common_samples]
else:
    common_samples = prot_samples
    print("SCOT+ pseudo-paired sample axes match exactly.")

# Final safety check
if not all(proteomics_mat.columns == transcriptomics_mat.columns):
    raise ValueError("Sample order mismatch between SCOT-aligned proteomics and transcriptomics.")

print("Pseudo-paired SCOT+ matrices ready for MOFA+.")
print(f"Proteomics matrix shape after checks: {proteomics_mat.shape}")
print(f"Transcriptomics matrix shape after checks: {transcriptomics_mat.shape}")

Proteomics samples: 68
Transcriptomics samples: 68
SCOT+ pseudo-paired sample axes match exactly.
Pseudo-paired SCOT+ matrices ready for MOFA+.
Proteomics matrix shape after checks: (1500, 68)
Transcriptomics matrix shape after checks: (1500, 68)


In [11]:
# Convert to long format
proteomics_long = (proteomics_mat.reset_index().melt(id_vars="Gene", var_name="sample", value_name="value").rename(columns={"Gene": "feature"}))
proteomics_long["view"] = "Proteomics"
proteomics_long["feature"] = proteomics_long["feature"].astype(str) + "_prot"

transcriptomics_long = (transcriptomics_mat.reset_index().melt(id_vars="Gene", var_name="sample", value_name="value")
                        .rename(columns={"Gene": "feature"}))
transcriptomics_long["view"] = "Transcriptomics"
transcriptomics_long["feature"] = transcriptomics_long["feature"].astype(str) + "_rna"

mofa_long = pd.concat([proteomics_long, transcriptomics_long], ignore_index=True)

print("\nMOFA long-format preview:")
print(mofa_long.head())
print("\nView counts:")
print(mofa_long["view"].value_counts())


MOFA long-format preview:
        feature       sample     value        view
0   Eif2b4_prot  05J_C10_A10 -2.272985  Proteomics
1  Slc38a2_prot  05J_C10_A10 -0.186504  Proteomics
2     Wnk1_prot  05J_C10_A10  0.526811  Proteomics
3    Krt6a_prot  05J_C10_A10 -1.336014  Proteomics
4  Rabgef1_prot  05J_C10_A10  0.286761  Proteomics

View counts:
view
Proteomics         102000
Transcriptomics    102000
Name: count, dtype: int64


In [13]:
ent.set_data_df(mofa_long, likelihoods=["gaussian", "gaussian"])


No "group" column found in the data frame, we will assume a common group for all samples...


Loaded group='single_group' view='Proteomics' with N=68 samples and D=1500 features...
Loaded group='single_group' view='Transcriptomics' with N=68 samples and D=1500 features...




In [15]:
ent.set_model_options(factors=5)

Model options:
- Automatic Relevance Determination prior on the factors: False
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Likelihoods:
- View 0 (Proteomics): gaussian
- View 1 (Transcriptomics): gaussian




In [17]:
ent.set_train_options(convergence_mode="slow", dropR2=0.001, gpu_mode=True, seed=1)


GPU mode is activated



In [ ]:
ent.build()
ent.run()

In [ ]:
out_model = os.path.join(OUTPUT_FOLDER, "SCOTplus_trained_model_MOFAcomp.hdf5")
ent.save(outfile=out_model)